# Node Type Classification — building the training data

The merge GNN currently learns edges only. Two failure modes motivate a second head:
AIS calls some **background** regions cells, and candidate edges get drawn between
**epithelial** and **hyphal** masks, which cannot merge.

The model will predict a per-node type (background / epithelial / hyphal) from the node
embeddings alongside the existing edge classifier, trained with a combined loss so it learns
a joint representation. Nothing is fed in explicitly — that a true edge cannot span
background↔cell or epithelial↔hyphal is left for the network to learn.

**This notebook builds and inspects the training labels. It does not train.**

Both label sets come off one decision, the background/cell split:

1. Choose `min_overlap_frac`, separating **background** fragments from **cell** fragments
   (§2). One threshold, chosen by looking at the overlay.
2. **Node-type labels** — background fragments are class `background`; the rest inherit
   their GT cell's type (**epithelial** / **hyphal**) from its shape, per-image because
   magnification differs (§3).
3. **Edge labels** — the per-GT-cell MST over the *cell* fragments, from the same split
   (§4). This is `cell_merge_labels(..., min_overlap_frac=<chosen>)`.

So the threshold feeds node types *and* edge labels. Raising it moves fragments out of their
GT cell, which both relabels them background and deletes the true edges they had to their
cell-mates.

In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Load images, AIS predictions, and GT masks

Same data config as `10_Merge Oversegmentation GNN.ipynb`. Embeddings are not needed to
build labels, so nothing is stitched here and the load stays fast.

In [ ]:
import json
from pathlib import Path

import numpy as np
import tifffile

from image_processing_tools.image_class.image_container import ImageContainer

DATA_CONFIG = "merge_oversegmentation_data.json"

with open(DATA_CONFIG) as f:
    SAMPLES = json.load(f)["samples"]


def _config(correct_dic_shift):
    """Per-sample preprocessing config. Mirrors notebook 10 so the fragments here are
    identical to the ones the merge GNN trains on."""
    return {
        "preprocessing": {
            "resize_image": False,
            "max_dim": 1080,
            "outlier_percentile": 0.35,
            "quantization": "16bit",
            "correct_DIC_shift": correct_dic_shift,
        }
    }


def load_label_map(path):
    """Read an instance label map as int32 (H, W)."""
    lab = np.squeeze(tifffile.imread(str(Path(path).expanduser())))
    assert lab.ndim == 2, f"Expected a 2D label map, got shape {lab.shape} for {path}"
    return lab.astype(np.int32)


ais_labels, gt_labels, intensity_images = [], [], []
for s in SAMPLES:
    channels = [Path(p).expanduser() for p in s["image"].values()]
    config = _config(s.get("correct_DIC_shift", [0, 0]))

    # Grouped -> _sum_channels reduces to one 2D channel. Used as the backdrop for every
    # figure here: it is always 2D, so it renders grayscale and never competes with the
    # overlay colors the way the multi-channel composite does (DIC lands in red there).
    # It is also the image the intensity features are read from.
    intensity_images.append(ImageContainer([channels], config).merge())

    ais_labels.append(load_label_map(s["ais"]))
    gt_labels.append(load_label_map(s["label"]))

print(f"Loaded {len(ais_labels)} images.")
for i, (a, g, im) in enumerate(zip(ais_labels, gt_labels, intensity_images)):
    assert a.shape == g.shape, f"img {i}: AIS {a.shape} != GT {g.shape}"
    assert im.ndim == 2 and im.shape == a.shape, f"img {i}: intensity {im.shape} != AIS {a.shape}"
    print(f"  img {i}: AIS fragments={len(np.unique(a))-1:>4}  GT cells={len(np.unique(g))-1:>3}  "
          f"channels={list(SAMPLES[i]['image'])}")

In [ ]:
# Thesis figures from this notebook are collected under one folder. save_fig writes a
# single PNG per figure; the folder's README documents what each one shows.
from pathlib import Path

FIG_DIR = Path("../../thesis_figures/node_type_classification")
FIG_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name, dpi=100):
    """Save `fig` as FIG_DIR/<name>.png at publication resolution."""
    fig.savefig(FIG_DIR / f"{name}.png", dpi=dpi, bbox_inches="tight")
    return fig


## 2. The background / cell split

`assign_fragments_to_gt` assigns each fragment to the GT cell it overlaps most, returning
`-1` when that overlap is below `min_overlap_frac` of the **fragment's own area**. The `-1`
nodes are background; the rest are cell fragments and go on to carry a type and to form
edges.

The sweep is the numeric view. Pick the threshold from the overlay in the next cell, not
from the table alone.

In [ ]:
from skimage.measure import regionprops

from image_processing_tools.scene_graph_network.cell_merge_labels import assign_fragments_to_gt
from image_processing_tools.scene_graph_network.gnn_interpret import plot_overlap_histogram

SWEEP = [0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.7]

print("Background node % by min_overlap_frac\n")
print(f"{'frac':<7}" + "".join(f"{'img'+str(i):>8}" for i in range(len(SAMPLES))) + f"{'ALL':>9}")
print("-" * (7 + 8 * len(SAMPLES) + 9))
for frac in SWEEP:
    line, tot_bg, tot_n = f"{frac:<7}", 0, 0
    for a, g in zip(ais_labels, gt_labels):
        assign = assign_fragments_to_gt(a, g, min_overlap_frac=frac)
        nb, n = int((assign == -1).sum()), len(assign)
        tot_bg += nb; tot_n += n
        line += f"{100*nb/n:>7.1f}%"
    print(line + f"{100*tot_bg/tot_n:>8.1f}%")

# Where the fragments actually sit. A cutoff in a sparse region of this histogram moves few
# fragments; one in a dense region is a coin flip for many -- and every fragment it moves
# also loses its true edges.
overlaps = []
for a, g in zip(ais_labels, gt_labels):
    for p in regionprops(a):
        gv = g[p.coords[:, 0], p.coords[:, 1]]
        gv = gv[gv != 0]
        overlaps.append(0.0 if gv.size == 0 else np.unique(gv, return_counts=True)[1].max() / p.area)
overlaps = np.array(overlaps)

# Proper display figure for the thesis (replaces the earlier text histogram). Saved into
# the shared thesis-figure folder for this notebook (see save_fig above).
fig = plot_overlap_histogram(overlaps, threshold=0.1)
save_fig(fig, "overlap_histogram")
fig

### Visual check

Edit `BACKGROUND_THRESH`, re-run, and look at what flips. Green = assigned to a GT cell,
red = background. A red fragment sitting on a real cell is one this cutoff is discarding
wrongly — it will be labeled background *and* the true merge edges it had to its
cell-mates will be deleted.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from image_processing_tools.scene_graph_network.gnn_train import _imshow_microscopy

# === EDIT ME ===
BACKGROUND_THRESH = 0.1
DOWNSAMPLE = 2          # display only; labels are computed at full resolution
ALPHA = 0.35            # overlay opacity; lower it to read the image under the masks


def fragment_class_rgba(ais, assign, downsample=1, alpha=ALPHA):
    """RGBA overlay: green where a fragment is assigned to a GT cell, red where background.

    `assign` is in regionprops order, which is label-ascending, so a LUT indexed by label
    value maps it straight back onto the pixels.
    """
    labels = [p.label for p in regionprops(ais)]
    lut = np.zeros(int(ais.max()) + 1, dtype=np.uint8)   # 0 = not a fragment
    for lab, gt_id in zip(labels, assign):
        lut[lab] = 1 if gt_id == -1 else 2               # 1 = background, 2 = cell
    cls = lut[ais][::downsample, ::downsample]

    rgba = np.zeros((*cls.shape, 4), dtype=np.float32)
    rgba[cls == 1] = (1.0, 0.15, 0.15, alpha)            # background -> red
    rgba[cls == 2] = (0.15, 1.0, 0.30, alpha)            # cell       -> green
    return rgba


n = len(ais_labels)
ncol = (n + 1) // 2
fig, axes = plt.subplots(2, ncol, figsize=(7 * ncol, 7.5 * 2), layout="constrained")
for i, ax in enumerate(axes.ravel()):
    if i >= n:
        ax.axis("off"); continue
    ais = ais_labels[i]
    assign = assign_fragments_to_gt(ais, gt_labels[i], min_overlap_frac=BACKGROUND_THRESH)

    # Grayscale backdrop on purpose: the red overlay is unreadable over the multi-channel
    # composite, where DIC occupies the red channel.
    _imshow_microscopy(ax, intensity_images[i], downsample=DOWNSAMPLE)
    ax.imshow(fragment_class_rgba(ais, assign, DOWNSAMPLE))

    n_bg = int((assign == -1).sum())
    ax.set_title(f"img {i} — {n_bg}/{len(assign)} background ({100*n_bg/len(assign):.0f}%)",
                 fontsize=18)
    ax.axis("off")

fig.suptitle(f"Fragment assignment at min_overlap_frac = {BACKGROUND_THRESH}", fontsize=24)
fig.legend(handles=[Line2D([0], [0], marker="s", color="none", markerfacecolor=c,
                           markeredgecolor="gray", markersize=12, label=l)
                    for c, l in [("#26ff4d", "assigned to a GT cell"),
                                 ("#ff2626", "background")]],
           loc="outside lower center", ncol=2, fontsize=16, frameon=False)
save_fig(fig, "fragment_assignment")
plt.show()

### Zoom on what the threshold moves

The full-frame view is too coarse to judge borderline fragments. This crops to the
fragments whose overlap falls between two thresholds — the ones the choice actually
decides. Everything shown is background at `HIGH` and a cell fragment at `LOW`.

In [ ]:
# === EDIT ME ===
LOW, HIGH = 0.1, 0.2     # inspect fragments this choice disagrees about
MAX_PANELS = 12
PAD = 40                 # px of context around each fragment

rows = []
for i, (ais, gt) in enumerate(zip(ais_labels, gt_labels)):
    lo = assign_fragments_to_gt(ais, gt, min_overlap_frac=LOW)
    hi = assign_fragments_to_gt(ais, gt, min_overlap_frac=HIGH)
    for j, p in enumerate(regionprops(ais)):
        if lo[j] != -1 and hi[j] == -1:          # cell at LOW, background at HIGH
            gv = gt[p.coords[:, 0], p.coords[:, 1]]
            gv = gv[gv != 0]
            frac = 0.0 if gv.size == 0 else np.unique(gv, return_counts=True)[1].max() / p.area
            rows.append((i, p.label, p.bbox, frac))

print(f"{len(rows)} fragments are cell@{LOW} but background@{HIGH} "
      f"-- these are what the choice decides.")
rows.sort(key=lambda r: r[3])

if rows:
    show = rows[:MAX_PANELS]
    ncol = min(4, len(show))
    nrow = (len(show) + ncol - 1) // ncol
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 4.4 * nrow),
                             squeeze=False, layout="constrained")
    for ax, (i, lab, bbox, frac) in zip(axes.ravel(), show):
        r0, c0, r1, c1 = bbox
        r0, c0 = max(r0 - PAD, 0), max(c0 - PAD, 0)
        r1, c1 = min(r1 + PAD, ais_labels[i].shape[0]), min(c1 + PAD, ais_labels[i].shape[1])

        _imshow_microscopy(ax, intensity_images[i][r0:r1, c0:c1])

        frag = (ais_labels[i][r0:r1, c0:c1] == lab)
        frag_rgba = np.zeros((*frag.shape, 4), dtype=np.float32)
        frag_rgba[frag] = (1.0, 0.6, 0.0, 0.55)
        ax.imshow(frag_rgba)

        ax.set_title(f"img {i} frag {lab} — overlap {frac:.2f}", fontsize=16)
        ax.axis("off")
    for ax in axes.ravel()[len(show):]:
        ax.axis("off")
    fig.suptitle(f"Fragments in dispute between {LOW} and {HIGH} (orange = the fragment)",
                 fontsize=20)
    save_fig(fig, "disputed_fragments")
    plt.show()

## 3. Epithelial vs hyphal

A cell fragment inherits the type of the GT cell it was assigned to in §2, so the type is
decided **once per GT mask** and then propagated to its fragments. A fragment of a hypha is
often round on its own — only the whole GT cell carries the shape that separates the classes.

The rule is **per image**, because magnification differs. The data says so plainly: hyphae in
images 0/1 have a mean width of 17.5 / 23.4 px, while the coculture images split at 100–160 px.
No global cutoff survives that.

**Images 0 and 1 are hyphae-only and get `{"all": "hyphal"}`, never a threshold.** They have
no epithelial population, so any cutoff computed inside them would just slice the hyphal
distribution in half. Image 1 contains a GT mask at circularity 0.90 — as round as any
epithelial cell — and it is still a hypha.

### Metrics on offer

Each metric declares which side is hyphal, so a rule only carries a number.

| metric | hyphal when | scale | rationale |
|---|---|---|---|
| `mean_width` | **low** | px | area / major axis ≈ how thick the cell is. **Best on this data** — see below. |
| `minor_axis` | low | px | The other way to measure width. Similar, but sensitive to a single bulge. |
| `extent` | low | — | area / bbox area. Good, but calls large *spiky* epithelial cells hyphal. |
| `axis_ratio` | high | — | major/minor. The obvious reading of "elongated", but hyphal axis-ratio spans 1.2–16, so a cutoff lands *inside* the hyphal population. |
| `solidity` | low | — | area / convex-hull area. Best 2-means separation, but visibly worse than `extent`. |
| `circularity` | low | — | 4πA/P². |
| `eccentricity` | high | — | Saturates near 1.0 for anything elongated; median 0.97 across all GT cells. Weakest. |

**Why `mean_width`.** The others measure *elongation* or *convexity*, and a spread-out epithelial
cell scores like a filament on both — that is precisely how `extent` and `solidity` mislabel the
large spiky cells in images 3 and 4. What actually makes a hypha a hypha is being **thin
everywhere**, which is what `mean_width` measures. Note it is in pixels, so it is scale-dependent
— fine here because the threshold is per-image, but it is the metric most sensitive to
magnification.

Do not trust the separation score alone: `solidity` scores best under 2-means and still loses on
inspection. Read the figure.

In [ ]:
from scipy.cluster.vq import kmeans2

# name -> (fn(regionprop) -> float, which side is hyphal)
# The direction lives with the metric so a rule only carries a number, and switching metric
# cannot silently invert the two classes.
CELL_TYPE_METRICS = {
    # Thinness. A hypha is thin everywhere; a spread-out epithelial cell is not, however
    # elongated or non-convex its outline. In pixels, so scale-dependent -- fine only
    # because the threshold is per-image.
    "mean_width":   (lambda p: p.area / max(p.major_axis_length, 1e-9), "low"),
    "minor_axis":   (lambda p: p.minor_axis_length, "low"),
    # Elongation / convexity. Dimensionless, but each confuses a spiky epithelial cell for
    # a filament to some degree.
    "extent":       (lambda p: p.extent, "low"),
    "axis_ratio":   (lambda p: p.major_axis_length / max(p.minor_axis_length, 1e-9), "high"),
    "solidity":     (lambda p: p.solidity, "low"),
    "circularity":  (lambda p: 4 * np.pi * p.area / max(p.perimeter ** 2, 1e-9), "low"),
    "eccentricity": (lambda p: p.eccentricity, "high"),
}

HYPHAE_ONLY = [0, 1]      # contain no epithelial cells -> never thresholded


def gt_metric_values(gt, metric):
    """(labels, values) for every GT cell under `metric`, in regionprops order."""
    fn, _ = CELL_TYPE_METRICS[metric]
    props = regionprops(gt)
    return (np.array([p.label for p in props]),
            np.array([fn(p) for p in props]))


def suggest_threshold(gt, metric):
    """Midpoint of a 2-means split on the metric — a starting point, not an answer.

    Meaningless where only one cell type is present, which is why HYPHAE_ONLY images are
    excluded rather than fitted: 2-means returns two clusters even when given one.
    """
    _, v = gt_metric_values(gt, metric)
    if len(v) < 2:
        return float("nan")
    cent, _ = kmeans2(v.reshape(-1, 1).astype(np.float64), 2, minit="++", seed=0)
    return float(np.mean(np.sort(cent.ravel())))


# === EDIT ME: any key of CELL_TYPE_METRICS ===
METRIC = "mean_width"

_, direction = CELL_TYPE_METRICS[METRIC]
print(f"metric = {METRIC!r}   (a GT cell is hyphal when its {METRIC} is {direction})\n")
print(f"{'img':<5}{'n':>4}{'min':>8}{'p10':>8}{'p25':>8}{'p50':>8}{'p75':>8}{'p90':>8}{'max':>8}"
      f"{'suggest':>10}   note")
print("-" * 80)
for i, gt in enumerate(gt_labels):
    _, v = gt_metric_values(gt, METRIC)
    q = np.percentile(v, [10, 25, 50, 75, 90])
    if i in HYPHAE_ONLY:
        sug, note = f"{'—':>10}", "hyphae only -> all hyphal"
    else:
        sug, note = f"{suggest_threshold(gt, METRIC):>10.2f}", "coculture"
    print(f"{i:<5}{len(v):>4}{v.min():>8.2f}" + "".join(f"{x:>8.2f}" for x in q)
          + f"{v.max():>8.2f}{sug}   {note}")

print("\n`suggest` is the midpoint of a 2-means split — seed CELL_TYPE_RULE with it, then\n"
      "check the figure. The split is unsupervised and has no idea what a cell is: on this\n"
      "data it ranks `solidity` best, which then visibly mislabels filaments in image 3.")
if METRIC in ("mean_width", "minor_axis"):
    print(f"\nNote: {METRIC} is in pixels. Compare the hyphae-only rows (images 0/1) against\n"
          "the coculture suggestions to see the magnification gap that forces a per-image rule.")

### Tune the per-image thresholds

`CELL_TYPE_RULE` holds one rule per image. Edit the numbers, re-run, and read the GT masks:
blue = epithelial, orange = hyphal. Images 0/1 carry `{"all": "hyphal"}` and ignore the
metric entirely.

This colors **GT cells**, not fragments — the type is decided once per GT mask here, and
propagated to fragments in the next cell.

In [ ]:
# === EDIT ME — one rule per image ===
# {"all": "hyphal"}                      -> every GT cell is hyphal, metric ignored
# {"metric": <name>, "threshold": <num>} -> hyphal on the side the metric declares
# Settled by reading the GT-type figure, seeded from the 2-means suggestions. Image 4
# needed tightening from the suggested 119.6 to 100.6; the rest stand as suggested.
CELL_TYPE_RULE = {
    0: {"all": "hyphal"},
    1: {"all": "hyphal"},
    2: {"metric": "mean_width", "threshold": 137.5},
    3: {"metric": "mean_width", "threshold": 101.0},
    4: {"metric": "mean_width", "threshold": 100.6},
    5: {"metric": "mean_width", "threshold": 159.8},
}

TYPE_COLORS = {"epithelial": (0.20, 0.45, 1.0), "hyphal": (1.0, 0.55, 0.0)}


def gt_cell_types(gt, rule):
    """{gt_label: "epithelial" | "hyphal"} for every GT cell in the image."""
    if "all" in rule:
        return {int(p.label): rule["all"] for p in regionprops(gt)}
    _, direction = CELL_TYPE_METRICS[rule["metric"]]
    labels, vals = gt_metric_values(gt, rule["metric"])
    hyphal = vals > rule["threshold"] if direction == "high" else vals < rule["threshold"]
    return {int(l): ("hyphal" if h else "epithelial") for l, h in zip(labels, hyphal)}


def type_rgba(label_map, type_of, downsample=1, alpha=ALPHA):
    """RGBA overlay coloring each labeled region by its type name."""
    lut = np.zeros((int(label_map.max()) + 1, 4), dtype=np.float32)
    for lab, t in type_of.items():
        lut[lab] = (*TYPE_COLORS[t], alpha)
    return lut[label_map[::downsample, ::downsample]]


n = len(gt_labels)
ncol = (n + 1) // 2
fig, axes = plt.subplots(2, ncol, figsize=(7 * ncol, 7.5 * 2), layout="constrained")
counts = {}
for i, ax in enumerate(axes.ravel()):
    if i >= n:
        ax.axis("off"); continue
    rule = CELL_TYPE_RULE[i]
    types = gt_cell_types(gt_labels[i], rule)
    counts[i] = {t: sum(1 for v in types.values() if v == t) for t in TYPE_COLORS}

    _imshow_microscopy(ax, intensity_images[i], downsample=DOWNSAMPLE)
    ax.imshow(type_rgba(gt_labels[i], types, DOWNSAMPLE))

    desc = "all hyphal" if "all" in rule else f"{rule['metric']} @ {rule['threshold']}"
    ax.set_title(f"img {i} — {counts[i]['epithelial']} epithelial, {counts[i]['hyphal']} hyphal"
                 f"\n({desc})", fontsize=18)
    ax.axis("off")

fig.suptitle("GT cell types", fontsize=24)
fig.legend(handles=[Line2D([0], [0], marker="s", color="none", markerfacecolor=c,
                           markeredgecolor="gray", markersize=12, label=l)
                    for l, c in TYPE_COLORS.items()],
           loc="outside lower center", ncol=2, fontsize=16, frameon=False)
save_fig(fig, "gt_cell_types")
plt.show()

print("GT cells per class:")
for i, c in counts.items():
    print(f"  img {i}: epithelial={c['epithelial']:>3}  hyphal={c['hyphal']:>3}")
tot_e = sum(c["epithelial"] for c in counts.values())
tot_h = sum(c["hyphal"] for c in counts.values())
print(f"  ALL  : epithelial={tot_e:>3}  hyphal={tot_h:>3}")

### Comparing the shape metrics

One metric per row, one coculture image per column. Each panel recolors the GT masks by
that metric's own hyphal/epithelial call, so a metric that mislabels cells shows it on
the image. `mean_width` uses the chosen per-image threshold; every other metric uses the
2-means suggested threshold, printed under each panel. The `mean_width` row separates the
two classes cleanly, while the elongation and convexity metrics mislabel spread-out
epithelial cells as filaments.

In [ ]:
# How each shape metric separates hyphal from epithelial, one metric per row and one
# coculture image per column. Each panel recolors the GT masks by that metric's own
# hyphal/epithelial call, so a metric that mislabels cells shows it directly on the image
# (blue = epithelial, orange = hyphal). mean_width uses the chosen per-image threshold
# from CELL_TYPE_RULE; every other metric uses the 2-means suggested threshold, printed
# under each panel. Only the metrics discussed in the thesis are shown.
METRIC_ROWS = ["mean_width", "minor_axis", "extent", "axis_ratio", "solidity", "eccentricity"]
COCULTURE = [i for i in range(len(gt_labels)) if i not in HYPHAE_ONLY]


def metric_threshold(i, metric):
    """Threshold for image i under metric: the chosen value for mean_width, else 2-means."""
    rule = CELL_TYPE_RULE[i]
    if metric == "mean_width" and rule.get("metric") == "mean_width":
        return rule["threshold"]
    return suggest_threshold(gt_labels[i], metric)


def metric_types(gt, metric, thresh):
    """{gt_label: 'epithelial'|'hyphal'} classifying each GT cell by `metric` at `thresh`."""
    _, direction = CELL_TYPE_METRICS[metric]
    labels, vals = gt_metric_values(gt, metric)
    hyphal = vals > thresh if direction == "high" else vals < thresh
    return {int(l): ("hyphal" if h else "epithelial") for l, h in zip(labels, hyphal)}


nrow, ncol = len(METRIC_ROWS), len(COCULTURE)
fig, axes = plt.subplots(nrow, ncol, figsize=(4.4 * ncol, 4.8 * nrow),
                         squeeze=False, layout="constrained")
for r, metric in enumerate(METRIC_ROWS):
    _, direction = CELL_TYPE_METRICS[metric]
    for c, i in enumerate(COCULTURE):
        ax = axes[r][c]
        thr = metric_threshold(i, metric)
        types = metric_types(gt_labels[i], metric, thr)
        _imshow_microscopy(ax, intensity_images[i], downsample=DOWNSAMPLE)
        ax.imshow(type_rgba(gt_labels[i], types, DOWNSAMPLE))
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values():
            s.set_visible(False)
        ax.set_xlabel(f"threshold = {thr:.1f}", fontsize=15)
        if r == 0:
            ax.set_title(f"img {i}", fontsize=18)
        if c == 0:
            src = "chosen" if metric == "mean_width" else "2-means"
            ax.set_ylabel(f"{metric}\n(hyphal {direction}, {src})", fontsize=16)

fig.suptitle("Cell-type shape metrics: how each rule labels the GT cells "
             "(blue = epithelial, orange = hyphal)", fontsize=22)
fig.legend(handles=[Line2D([0], [0], marker="s", color="none", markerfacecolor=c,
                           markeredgecolor="gray", markersize=12, label=l)
                    for l, c in TYPE_COLORS.items()],
           loc="outside lower center", ncol=2, fontsize=16, frameon=False)
save_fig(fig, "cell_type_metrics")
plt.show()


### Node labels

Combine the two decisions into the per-node target the classifier trains on:

- background fragment (§2, `assign == -1`) → **background**
- otherwise → the type of the GT cell it was assigned to (§3)

These are the labels the node head learns. Per-class counts matter here: under leave-one-out
CV a fold can train on images that have epithelial cells and be tested on one that has none,
so a class that is rare overall may be absent entirely from a given fold.

In [ ]:
# Class ids the node head will predict. Background is 0 so it stays the "reject" class.
NODE_CLASSES = {"background": 0, "epithelial": 1, "hyphal": 2}
NODE_CLASS_NAMES = {v: k for k, v in NODE_CLASSES.items()}
NODE_COLORS = {"background": (1.0, 0.15, 0.15),
               "epithelial": (0.20, 0.45, 1.0),
               "hyphal":     (1.0, 0.55, 0.0)}


def node_type_labels(ais, gt, rule, min_overlap_frac):
    """(N,) int64 node-type targets in regionprops order, matching the graph's node order.

    A fragment's type comes from the GT cell it was ASSIGNED to, not from its own shape: a
    fragment of a hypha is often round on its own, and only the whole GT cell carries the
    shape that separates the classes. That is what makes this a learning problem rather
    than a restatement of the node features.
    """
    assign = assign_fragments_to_gt(ais, gt, min_overlap_frac=min_overlap_frac)
    types = gt_cell_types(gt, rule)
    out = np.zeros(len(assign), dtype=np.int64)
    for i, g in enumerate(assign):
        out[i] = NODE_CLASSES["background"] if g == -1 else NODE_CLASSES[types[int(g)]]
    return out


node_types = [
    node_type_labels(ais_labels[i], gt_labels[i], CELL_TYPE_RULE[i], BACKGROUND_THRESH)
    for i in range(len(ais_labels))
]

n = len(ais_labels)
ncol = (n + 1) // 2
fig, axes = plt.subplots(2, ncol, figsize=(7 * ncol, 7.5 * 2), layout="constrained")
for i, ax in enumerate(axes.ravel()):
    if i >= n:
        ax.axis("off"); continue
    # regionprops order is label-ascending, so a LUT indexed by label value maps the
    # per-node classes straight back onto the fragment pixels.
    labels = [p.label for p in regionprops(ais_labels[i])]
    type_of = {int(l): NODE_CLASS_NAMES[int(t)] for l, t in zip(labels, node_types[i])}

    lut = np.zeros((int(ais_labels[i].max()) + 1, 4), dtype=np.float32)
    for lab, t in type_of.items():
        lut[lab] = (*NODE_COLORS[t], ALPHA)

    _imshow_microscopy(ax, intensity_images[i], downsample=DOWNSAMPLE)
    ax.imshow(lut[ais_labels[i][::DOWNSAMPLE, ::DOWNSAMPLE]])

    c = np.bincount(node_types[i], minlength=3)
    ax.set_title(f"img {i} — bg {c[0]} | epi {c[1]} | hyph {c[2]}", fontsize=18)
    ax.axis("off")

fig.suptitle(f"Node-type labels  (background @ {BACKGROUND_THRESH})", fontsize=24)
fig.legend(handles=[Line2D([0], [0], marker="s", color="none", markerfacecolor=c,
                           markeredgecolor="gray", markersize=12, label=l)
                    for l, c in NODE_COLORS.items()],
           loc="outside lower center", ncol=3, fontsize=16, frameon=False)
save_fig(fig, "node_type_labels")
plt.show()

print(f"{'img':<6}{'background':>12}{'epithelial':>12}{'hyphal':>9}{'total':>8}")
print("-" * 47)
total = np.zeros(3, dtype=int)
for i, t in enumerate(node_types):
    c = np.bincount(t, minlength=3)
    total += c
    print(f"{i:<6}{c[0]:>12}{c[1]:>12}{c[2]:>9}{c.sum():>8}")
print("-" * 47)
print(f"{'ALL':<6}{total[0]:>12}{total[1]:>12}{total[2]:>9}{total.sum():>8}")
print("       " + "".join(f"{100*x/total.sum():>11.1f}%" for x in total))

## 4. Edge labels from the same split

`cell_merge_labels(ais, gt, min_overlap_frac=BACKGROUND_THRESH)` takes the *same* background
split as §2 and returns the per-GT-cell MST over the cell fragments — a chain or tree per
cell, never a clique. Those are the positive edges; every other candidate edge is negative.

Candidates themselves come from `extract_cell_graph`, which builds kNN edges by
boundary-to-boundary distance over **every** fragment, background included — nothing filters
them out. That is what makes the joint task learnable: the graph already contains the
counterexamples.

The breakdown below classifies every candidate edge by the types of its two endpoints. It
answers whether the two constraints the node head is meant to internalise —
background↮cell and epithelial↮hyphal — actually have negatives to learn from.

In [ ]:
from itertools import combinations_with_replacement

from image_processing_tools.scene_graph_network.cell_mask_graph import extract_cell_graph
from image_processing_tools.scene_graph_network.cell_merge_labels import cell_merge_labels

K = 10   # kNN candidate edges per fragment. Notebooks 10 and 12 both use 10 as well.
# k=7 leaves 22/483 true merges (4.6%) outside the candidate set, where no model can ever
# predict them; k=10 cuts that to 6/483 (1.2%).

# `extract_cell_graph` returns one entry per unordered pair; `build_cell_graph_data` then
# emits both directions, which is what the model sees and what notebook 10 prints. Counts
# below are DIRECTED (= pairs x 2) so they line up with notebook 10 -- every pair is present
# in both directions, so the doubling is exact rather than an estimate.
edge_index_list, edge_labels = [], []
missed = []
for i in range(len(ais_labels)):
    # Candidates over every fragment, background included. Does not depend on
    # BACKGROUND_THRESH -- the threshold moves labels, never the candidate set.
    _, _, _, _, ei = extract_cell_graph(ais_labels[i], intensity_images[i], k=K)
    cand = {(min(u, v), max(u, v)) for u, v in zip(ei[0], ei[1])}
    mst = set(cell_merge_labels(ais_labels[i], gt_labels[i], min_overlap_frac=BACKGROUND_THRESH))

    lbl = np.array([1.0 if (min(u, v), max(u, v)) in cand & mst else 0.0
                    for u, v in zip(ei[0], ei[1])], dtype=np.float32)
    edge_index_list.append(np.asarray(ei))
    edge_labels.append(lbl)
    missed.append((len(mst - cand), len(mst)))
    print(f"graph {i}: {len(node_types[i]):>3} nodes, {len(lbl)*2:>5} directed edges, "
          f"{int(lbl.sum())*2:>4} positive-edge entries")

# Every unordered pair of node classes, e.g. ("background", "hyphal").
PAIRS = [tuple(sorted(p)) for p in
         combinations_with_replacement(sorted(NODE_CLASSES, key=NODE_CLASSES.get), 2)]
SHORT = {"background": "bg", "epithelial": "epi", "hyphal": "hyph"}
pair_name = lambda p: f"{SHORT[p[0]]}-{SHORT[p[1]]}"

rows = []
totals = {p: 0 for p in PAIRS}
total_pos = total_edges = 0
for i in range(len(ais_labels)):
    c = {p: 0 for p in PAIRS}
    n_pos = 0
    for k, (u, v) in enumerate(zip(*edge_index_list[i])):
        if edge_labels[i][k] == 1.0:
            n_pos += 2                  # positives are same-cell by construction
            continue
        key = tuple(sorted((NODE_CLASS_NAMES[int(node_types[i][u])],
                            NODE_CLASS_NAMES[int(node_types[i][v])])))
        c[key] += 2
    for p in PAIRS:
        totals[p] += c[p]
    total_pos += n_pos
    total_edges += len(edge_labels[i]) * 2
    rows.append((i, len(edge_labels[i]) * 2, n_pos, c))

hdr = f"{'img':<5}{'edges':>7}{'POS':>6}   " + "".join(f"{pair_name(p):>11}" for p in PAIRS)
print(f"\nNegative candidate edges by endpoint type pair (directed, k={K})\n")
print(hdr)
print("-" * len(hdr))
for i, n_e, n_pos, c in rows:
    print(f"{i:<5}{n_e:>7}{n_pos:>6}   " + "".join(f"{c[p]:>11}" for p in PAIRS))
print("-" * len(hdr))
print(f"{'ALL':<5}{total_edges:>7}{total_pos:>6}   " + "".join(f"{totals[p]:>11}" for p in PAIRS))

print(f"\nShare of all {total_edges} directed candidate edges:")
print(f"  {'positive (merge)':<28}{total_pos:>5}  ({100*total_pos/total_edges:>5.1f}%)")
for p in PAIRS:
    print(f"  {'negative ' + pair_name(p):<28}{totals[p]:>5}  ({100*totals[p]/total_edges:>5.1f}%)")

cell_bg = totals[("background", "epithelial")] + totals[("background", "hyphal")]
epi_hyph = totals[("epithelial", "hyphal")]
print("\nNegatives backing the two constraints the node head should internalise:")
print(f"  cell  <-> background : {cell_bg:>5}")
print(f"  epithelial <-> hyphal: {epi_hyph:>5}")
print("Concentrated in the coculture images: 0/1 contribute no epi-hyph edges at all, so a\n"
      "leave-one-out fold holding out image 3 or 4 removes a large share of them.")

# Recall ceiling. An MST pair that is not a kNN candidate is not labeled negative -- it is
# not an edge at all, so no model can ever predict that merge however good it gets.
m_miss, m_tot = (sum(x) for x in zip(*missed))
print(f"\nTrue merges unreachable because they are not kNN candidates (k={K}):")
for i, (miss, tot) in enumerate(missed):
    print(f"  img {i}: {miss:>3}/{tot:>3} missed")
print(f"  ALL  : {m_miss}/{m_tot} ({100*m_miss/max(m_tot,1):.1f}%) -> merge recall caps at "
      f"{100*(1-m_miss/max(m_tot,1)):.1f}%")